# Build your first enchilada Block

A Block contains a model and an algorithm for advancing that model. **The block owns
its sampler state; Wheel owns its accepted physical contribution.** By the end of this tutorial, you will have a working
sinusoid-amplitude sampler, a noise-only block, and checks that the blocks can
run across observation domains with separate instances for independent campaigns.

We will cover:

1. Decide what the block models and which inputs it supports.
2. Separate sampler state from published physical state.
3. Implement `sample` and construct a complete initial result.
4. Start at a known injection, run a campaign, and check the posterior.
5. Continue a run and choose a domain through Wheel.
6. Publish noise estimates through the same Block interface.

The signals are synthetic, with amplitudes in arbitrary units. This teaches the
exchange contract, not a physical LISA waveform or response model.

## Setup

From the repository root:

```sh
uv sync --extra examples
uv pip install matplotlib
uv run --no-sync jupyter lab examples/create_block.ipynb
```

Choose that environment's Python kernel and run the notebook top to bottom.
The computations need only enchilada and NumPy on Python 3.12 or newer. Matplotlib
is optional and used for the diagnostic figure. The main tutorial needs no WDM,
Eryn, GBGPU, or orbit package. Installing optional packages after `uv sync`, then
launching with `--no-sync`, preserves them.

In [ ]:
from copy import deepcopy
from dataclasses import dataclass, field

import numpy as np

from enchilada import (
    Block,
    BlockState,
    DataCovariance,
    L1Data,
    TranslatedCovariance,
    Wheel,
    transform,
)
from enchilada.testing import check_block

## 1. Understand the Block contract

A normal Python class qualifies as a `Block` when it has these two members; no
inheritance, decorator, or plugin registration is required.

| Member | When Wheel uses it | What it does |
| --- | --- | --- |
| `name` | Registration and ledger lookup | A unique, nonempty name within this Wheel. |
| `sample(conditional_residual, noise_covariance, current_block_state, *, rng)` | Once per block per cycle | Advance the model against the current inputs and return its complete physical state. |

The application constructs a complete initial `BlockState` and supplies it to
`wheel.add(block, initial_block_state=...)`. This argument is required and cannot
be `None`. Registration validates and adopts that result without calling model
code or consuming any Wheel RNG draws. Prior draws, injected values, or another
estimate are application choices, made before registration.

For block *i*, the conditional residual is

```text
observations − sum of every OTHER block's accepted TDI signal
```

Its own signal is still present. Evaluate this residual against your candidate
signal. Wheel performs subtraction between blocks; your block never subtracts
another block or adds its own old contribution back.

```text
application builds initial BlockState
                 ↓
wheel.add(block, initial_block_state=...) → validate → ledger + residual + covariance
                                                            ↓
                       conditional residual + covariance + current result + rng
                                                            ↓
                                                         sample
                                                            ↓
                       complete BlockState → validate → ledger + residual + covariance
```

Wheel passes independent physical snapshots and commits its RNG only with an
accepted result. A failed call leaves the published physical state and Wheel RNG
unchanged, but does not undo changes inside the block. Model code must manage
retries or restore its own `state`. Earlier successful blocks remain committed.
Orbit resources are shared and must be treated as immutable.


## 2. Specify the model before implementing the interface

Our model is `signal(t) = amplitude * sin(2*pi*frequency_hz*t)` on channel A.
Frequency is fixed. Amplitude has a Gaussian prior with configurable mean and
standard deviation. The likelihood is Gaussian with the supplied covariance.

The block accepts real time-domain data, channel A, and the
`fractional_frequency` convention. These are explicit tutorial choices. A real
source model should validate its own channel, observable, TDI, and orbit needs.
`L1Data` infers channel names from dictionary order; its settings are keyword-only.

| Ownership | Contents |
| --- | --- |
| Block configuration | Name, frequency, prior settings, proposal scale, proposals per call |
| Block-owned `State` | Proposal and acceptance counts, current log target, sampler diagnostics |
| Published `BlockState` | `template`, optional `covariance_matrix`, and physical `parameters` |

A counter is not a physical parameter. Here it demonstrates state carried across
calls inside the block's nested `State` dataclass. A larger sampler may also need
walkers, proposal adaptation, temperatures, or an external sampler's RNG state.
Wheel owns the supplied NumPy RNG; internal state is managed by the model.

In [ ]:
num_time_samples = 256
sample_rate_hz = 1.0
frequency_hz = 0.03125
true_amplitude = 1.4
noise_standard_deviation = 0.5
sample_times_s = np.arange(num_time_samples) / sample_rate_hz
signal_basis = np.sin(2 * np.pi * frequency_hz * sample_times_s)

observed = L1Data(
    {
        "A": true_amplitude * signal_basis
        + np.random.default_rng(11).normal(
            0.0, noise_standard_deviation, num_time_samples
        )
    },
    sample_rate_hz=sample_rate_hz,
    tdi_generation="2.0",
    physical_observable="fractional_frequency",
)
noise_covariance = DataCovariance.from_variance(
    observed, time_sample_variance=noise_standard_deviation**2
)
print(f"{observed.num_time_samples} samples; injected amplitude {true_amplitude}")

## 3. Choose what `sample` means for this model

We will use a small symmetric random-walk Metropolis sampler. Each call proposes
several amplitudes. For an accepted proposal, it advances to that amplitude; for
a rejected proposal, it retains the previous amplitude. Both cases return the
complete signal and state.

For this one-parameter Gaussian problem, the exact conditional is also Gaussian.
In real time coordinates, with basis `b`, conditional residual `r`, and covariance
`C`, its precision and mean are:

```text
precision = 1 / prior_std² + bᵀ C⁺ b
mean      = (prior_mean / prior_std² + bᵀ C⁺ r) / precision
```

`C⁺` is implemented by `noise_covariance.solve(...)`; it applies precision on the
retained subspace. Using the operator preserves correlations and statistical
exclusions, including those carried by a `TranslatedCovariance`. The proper prior
keeps the amplitude posterior well-defined even when the basis is excluded.
These ordinary dot products apply here because the block requests real time
coordinates; raw Fourier dot products need the appropriate real-coordinate weights.

The proposal is symmetric, so its acceptance probability is the smaller of one
and the posterior density ratio. Constants independent of amplitude, including
the covariance determinant, cancel in that ratio. A block sampling noise generally
needs the determinant as well.

This conditional could be drawn directly. We use Metropolis to teach continuation
and rejection handling, then check its output against the exact answer. Several
Metropolis steps are a transition targeting the conditional, not an independent
exact conditional draw.

## 4. Implement the complete block

Read the class in this order:

1. `__post_init__` checks fixed settings; the nested `State` dataclass
   owns the counters and diagnostics separately.
2. `_basis` validates the input conventions; `_render` builds a complete result.
3. `sample` reads amplitude from `current_block_state` and counters from
   `self.state`, refreshes the conditional target, and returns the next physical state.

A helper beginning with `_` is ordinary implementation code; Wheel only calls
`sample`. Application setup will supply the initial amplitude, rendered signal,
and counters separately. This entire class can be copied into your own module.

In [ ]:
@dataclass
class SineAmplitudeBlock:
    name: str
    frequency_hz: float
    prior_mean: float = 0.0
    prior_std: float = 2.0
    proposal_std: float = 0.08
    steps_per_call: int = 8

    @dataclass
    class State:
        num_proposals: int = 0
        num_acceptances: int = 0
        log_target: float | None = None
        sampler: str = "random-walk Metropolis"

    state: State = field(default_factory=State, init=False)

    def __post_init__(self):
        if not self.name:
            raise ValueError("Choose a nonempty block name")
        if not np.isfinite(self.frequency_hz) or self.frequency_hz <= 0:
            raise ValueError("frequency_hz must be finite and positive")
        if not np.isfinite(self.prior_mean):
            raise ValueError("prior_mean must be finite")
        if (
            not np.isfinite([self.prior_std, self.proposal_std]).all()
            or min(self.prior_std, self.proposal_std) <= 0
        ):
            raise ValueError("Prior and proposal standard deviations must be positive")
        if (
            isinstance(self.steps_per_call, bool)
            or not isinstance(self.steps_per_call, int)
            or self.steps_per_call < 1
        ):
            raise ValueError("steps_per_call must be a positive integer")

    def _basis(self, conditional_residual: L1Data) -> np.ndarray:
        if conditional_residual.data_domain != "time":
            raise ValueError("Register this block with data_domain='time'")
        if conditional_residual.channel_names != ("A",):
            raise ValueError("This tutorial block models only channel A")
        if conditional_residual.physical_observable != "fractional_frequency":
            raise ValueError("This tutorial expects fractional_frequency observations")
        if self.frequency_hz >= conditional_residual.sample_rate_hz / 2:
            raise ValueError("frequency_hz must be below Nyquist")
        times_s = (
            conditional_residual.start_time_gps
            + np.arange(conditional_residual.num_time_samples)
            / conditional_residual.sample_rate_hz
        )
        return np.sin(2 * np.pi * self.frequency_hz * times_s)

    def _render(self, conditional_residual, amplitude):
        return conditional_residual.block_state(
            template={"A": amplitude * self._basis(conditional_residual)},
            parameters={"amplitude": float(amplitude)},
        )

    def sample(
        self,
        conditional_residual: L1Data,
        noise_covariance: DataCovariance | TranslatedCovariance | None,
        current_block_state: BlockState,
        *,
        rng: np.random.Generator,
    ) -> BlockState:
        basis = self._basis(conditional_residual)
        if noise_covariance is None or noise_covariance.data_domain != "time":
            raise ValueError("This sampler needs a covariance in the time domain")
        noise_covariance.check_compatible(conditional_residual)
        precision_basis = noise_covariance.solve({"A": basis})["A"]
        precision_data = noise_covariance.solve(conditional_residual.channel_data)["A"]
        posterior_precision = 1 / self.prior_std**2 + float(basis @ precision_basis)
        posterior_mean = (
            self.prior_mean / self.prior_std**2 + float(basis @ precision_data)
        ) / posterior_precision

        def log_target(amplitude):
            return -0.5 * posterior_precision * (amplitude - posterior_mean) ** 2

        amplitude = float(current_block_state.parameters["amplitude"])
        num_proposals = self.state.num_proposals
        num_acceptances = self.state.num_acceptances
        # Other blocks may have changed the residual or covariance. Refresh the
        # current density for this call instead of caching it between calls.
        current_log_target = log_target(amplitude)
        for _ in range(self.steps_per_call):
            proposed_amplitude = rng.normal(amplitude, self.proposal_std)
            proposed_log_target = log_target(proposed_amplitude)
            acceptance_probability = np.exp(
                min(0.0, proposed_log_target - current_log_target)
            )
            num_proposals += 1
            if rng.random() < acceptance_probability:
                amplitude = proposed_amplitude
                current_log_target = proposed_log_target
                num_acceptances += 1

        result = self._render(conditional_residual, amplitude)
        self.state = self.State(num_proposals, num_acceptances, current_log_target)
        return result

### Why these result fields matter

`template` is the sum of all sources modeled by this block, on
its input grid. `conditional_residual.block_state(...)` checks the channel keys,
shape, and real/complex convention. Wheel additionally checks finiteness and
copies the result before adopting it.

A rejected proposal still returns the retained amplitude and its full signal.
**Returning `template=None` means the block now contributes zero**;
it is not shorthand for keeping the previous signal. The same complete-snapshot
rule applies to published physical parameters. These must support deep copying.
Sampler continuation and diagnostics stay on the block, whose internal `State`
can also hold live resources when the model needs them.

Our signal block omits `covariance_matrix`, which leaves the global covariance
unchanged because this block is not its owner.

## 5. Check physical and sampler state separately

`BlockState` publishes only `template`, `covariance_matrix`, and `parameters`. Its arrays
must match the input grid; `conditional_residual.block_state(...)` validates their
keys and shapes. Wheel validates and copies the physical state before adopting it.

The nested `SineAmplitudeBlock.State` holds counters and diagnostics on `self.state`.
The sampler builds its next physical output before replacing this internal state,
but Wheel can still reject a publication afterward. Applications control any
internal-state restoration needed for a retry.

In [ ]:
signal_block = SineAmplitudeBlock(name="signal", frequency_hz=frequency_hz)
assert isinstance(signal_block, Block)

initial_at_injection = observed.block_state(
    {"A": true_amplitude * signal_basis},
    parameters={"amplitude": true_amplitude},
)
# check_block executes the model. Give it a separate instance so its internal
# counters do not advance the block used in the main campaign.
probe = deepcopy(signal_block)
check_block(
    block_under_test=probe,
    observed_data=observed,
    initial_block_state=initial_at_injection,
    initial_noise_covariance=noise_covariance,
    num_cycles=3,
    random_seed=5,
)
assert probe.state.num_proposals == 3 * probe.steps_per_call
assert signal_block.state.num_proposals == 0
print("Initial parameters:", initial_at_injection.parameters)
print("Initial sampler state:", signal_block.state)

## 6. Register the injection as the campaign's starting estimate

Pass the complete result to `Wheel.add(initial_block_state=...)`. Wheel does not
call an initializer or draw a prior; registration consumes no Wheel RNG draws.
Our amplitude and rendered signal describe one consistent physical state;
proposal counters are initialized separately on the block. Model authors must check physical consistency and prior support;
Wheel can only check the exchange contract.

The initial signal uses the domain and WDM grid requested at registration, or the
observation grid if no domain is requested. Here both are time-domain. Wheel
validates, copies, and converts the result to its observation grid before adopting
it. Optional covariance can use its own grid metadata and follows the usual
conversion and ownership rules, including for a noise-only starting result.

The observations still contain the injection plus noise. The initial estimate
sets the ledger and residual; the declared prior stays the same and amplitude
continues to be sampled. This initializes one block; a full campaign checkpoint
would also need the other blocks' states and the Wheel RNG state.

A prior start is another application choice: draw an amplitude with a separate
setup RNG, render its signal, and initialize the same counters before calling
`add`. Such a helper belongs in application or model setup and is not a required
Block method. Keep the prior in the sampling target regardless of the start.


In [ ]:
wheel = Wheel(observed, initial_noise_covariance=noise_covariance, random_seed=23)
pristine_values = observed.channel_data["A"].copy()
wheel.add(signal_block, initial_block_state=initial_at_injection, data_domain="time")
assert wheel.ledger["signal"].parameters["amplitude"] == true_amplitude
assert signal_block.state.num_proposals == 0
assert signal_block.state.num_acceptances == 0
np.testing.assert_array_equal(
    wheel.contribution("signal")["A"], true_amplitude * signal_basis
)
np.testing.assert_array_equal(wheel.observed_data.channel_data["A"], pristine_values)
np.testing.assert_allclose(
    wheel.working_residual.channel_data["A"],
    pristine_values - true_amplitude * signal_basis,
)
print("Known-injection start checks passed")

## 7. Collect a posterior chain from the injection start

Continue with the initialized `wheel` above. The ledger stores the latest accepted
physical result, not sampler diagnostics or a history of every draw. Use `on_cycle_complete` to collect the chain
outside the block. Its callback runs after every block in that cycle is committed.

Below, each stored amplitude follows eight internal Metropolis proposals. Starting
at the injection does not fix the amplitude: every cycle advances it under the
same posterior target. We discard an initial warm-up segment before comparing
with the analytic posterior.


In [ ]:
amplitude_chain = []


def collect_amplitude(cycle_index, campaign):
    amplitude_chain.append(campaign.ledger["signal"].parameters["amplitude"])


num_cycles = 1200
warmup_cycles = 200
wheel.run(num_cycles=num_cycles, on_cycle_complete=collect_amplitude)
posterior_samples = np.asarray(amplitude_chain[warmup_cycles:])
last_result = wheel.ledger["signal"]
acceptance_fraction = (
    signal_block.state.num_acceptances / signal_block.state.num_proposals
)
assert signal_block.state.num_proposals == num_cycles * signal_block.steps_per_call
assert initial_at_injection.parameters == {"amplitude": true_amplitude}
print(
    f"Posterior amplitude: {posterior_samples.mean():.4f} ± {posterior_samples.std():.4f}"
)
print(f"Acceptance fraction: {acceptance_fraction:.3f}")

## 8. Check the model and sampler, not just the interface

The initial example uses known white noise. Independently compute the analytic
amplitude posterior from the raw arrays and its known variance, then compare the
sample mean and spread. These checks complement `check_block`.

For a new physical block, add injection/recovery checks, likelihood comparisons,
prior tests, convergence diagnostics, and checks of waveform conventions. The
small check here is appropriate for this one-parameter tutorial; it is not a
convergence guarantee for an arbitrary sampler.

In [ ]:
reference_precision = (
    1 / signal_block.prior_std**2
    + float(signal_basis @ signal_basis) / noise_standard_deviation**2
)
reference_mean = (
    signal_block.prior_mean / signal_block.prior_std**2
    + float(signal_basis @ observed.channel_data["A"]) / noise_standard_deviation**2
) / reference_precision
reference_std = np.sqrt(1 / reference_precision)

assert abs(posterior_samples.mean() - reference_mean) < 0.25 * reference_std
assert abs(posterior_samples.std() / reference_std - 1) < 0.20
assert abs(reference_mean - true_amplitude) < 5 * reference_std
np.testing.assert_allclose(
    wheel.working_residual.channel_data["A"],
    observed.channel_data["A"] - last_result.template["A"],
)
print(f"Analytic amplitude: {reference_mean:.4f} ± {reference_std:.4f}")
print("Posterior checks passed")

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None

if plt is None:
    print("Optional plots: install matplotlib in this kernel to display them.")
else:
    figure, axes = plt.subplots(1, 3, figsize=(14, 4), constrained_layout=True)
    axes[0].plot(amplitude_chain, linewidth=0.7)
    axes[0].axvline(warmup_cycles, color="grey", linestyle="--", label="End of warm-up")
    axes[0].axhline(reference_mean, color="black", linestyle=":", label="Analytic mean")
    axes[0].set(title="Amplitude chain", xlabel="Wheel cycle", ylabel="Amplitude")
    axes[0].legend()

    amplitude_grid = np.linspace(
        reference_mean - 4 * reference_std, reference_mean + 4 * reference_std, 200
    )
    reference_density = np.exp(
        -0.5 * ((amplitude_grid - reference_mean) / reference_std) ** 2
    ) / (reference_std * np.sqrt(2 * np.pi))
    axes[1].hist(
        posterior_samples, bins=30, density=True, alpha=0.65, label="Metropolis draws"
    )
    axes[1].plot(
        amplitude_grid, reference_density, color="black", label="Analytic posterior"
    )
    axes[1].axvline(
        true_amplitude, color="tab:red", linestyle="--", label="Injected value"
    )
    axes[1].set(title="Posterior check", xlabel="Amplitude", ylabel="Density")
    axes[1].legend()

    axes[2].plot(
        sample_times_s, observed.channel_data["A"], alpha=0.55, label="Observed A"
    )
    axes[2].plot(
        sample_times_s,
        posterior_samples.mean() * signal_basis,
        label="Posterior mean signal",
    )
    axes[2].set(
        title="Signal reconstruction", xlabel="Time from start [s]", ylabel="Amplitude"
    )
    axes[2].legend()
    plt.show()

## 9. Continue a run and keep campaigns independent

A second `run` continues the same Wheel and block instance. With the same seed,
initial physical state, internal block state, and inputs, two then three cycles
must match a single five-cycle run.

Use a separate block instance for each campaign: `self.state` belongs to that
instance. Restarting with a new instance requires explicitly restoring both the
published physical state and a copy of the model's internal `state`, along with
the campaign RNG. Ledger copies contain only the physical state.

In [ ]:
continued = Wheel(observed, noise_covariance, random_seed=31)
uninterrupted = Wheel(observed, noise_covariance, random_seed=31)
continued_block = SineAmplitudeBlock(name="signal", frequency_hz=frequency_hz)
uninterrupted_block = SineAmplitudeBlock(name="signal", frequency_hz=frequency_hz)
for campaign, block in (
    (continued, continued_block),
    (uninterrupted, uninterrupted_block),
):
    campaign.add(block, initial_block_state=initial_at_injection, data_domain="time")
continued.run(2)
continued.run(3)
uninterrupted.run(5)
continued_result = continued.ledger["signal"]
uninterrupted_result = uninterrupted.ledger["signal"]
assert continued_result.parameters == uninterrupted_result.parameters
assert continued_block.state == uninterrupted_block.state
assert continued_block.state is not uninterrupted_block.state
np.testing.assert_array_equal(
    continued_result.template["A"], uninterrupted_result.template["A"]
)
continued_result.parameters["amplitude"] = 999.0
assert continued.ledger["signal"].parameters != continued_result.parameters
print("Continuation and isolation checks passed")

## 10. Keep domain translation in Wheel

The same block can work when the campaign stores Fourier observations. Register
it with `data_domain="time"`; Wheel translates the conditional residual, covariance,
and previous signal for the call, then returns the accepted signal to the original
grid for storage. The initial signal uses the requested time-domain grid, even
though the observations are stored in frequency coordinates. Published parameters are copied without reinterpretation; the block-owned
`State` is not translated.

Our block never calls `transform`. The application selects the representation at
registration. The following example uses only NumPy and retains the same draws.

In [ ]:
spectrum = transform(observed, "frequency")
frequency_campaign = Wheel(
    spectrum,
    initial_noise_covariance=transform(noise_covariance, "frequency"),
    random_seed=31,
)
frequency_block = SineAmplitudeBlock(name="signal", frequency_hz=frequency_hz)
frequency_campaign.add(
    frequency_block, initial_block_state=initial_at_injection, data_domain="time"
)
frequency_campaign.run(5)
np.testing.assert_allclose(
    frequency_campaign.ledger["signal"].parameters["amplitude"],
    uninterrupted.ledger["signal"].parameters["amplitude"],
    atol=1e-12,
)
assert frequency_campaign.working_residual.data_domain == "frequency"
assert (
    frequency_campaign.contribution("signal")["A"].shape
    == spectrum.channel_data["A"].shape
)
np.testing.assert_allclose(
    transform(frequency_campaign.working_residual, "time").channel_data["A"],
    uninterrupted.working_residual.channel_data["A"],
    atol=1e-12,
)


# A genuinely translated time covariance can also be consumed via solve.
# Excluding Fourier DC corresponds to a nonlocal statistical exclusion in time.
@dataclass(frozen=True)
class FlatPSD:
    level: float

    def psd(self, frequencies, channel=None):
        return np.full_like(frequencies, self.level)


native_spectral_noise = DataCovariance.from_psd(
    spectrum, FlatPSD(2 * noise_standard_deviation**2 / sample_rate_hz)
)
# from_psd constructs frequency covariance and excludes DC by convention.
masked_time_noise = transform(native_spectral_noise, "time")
assert isinstance(masked_time_noise, TranslatedCovariance)
check_block(
    SineAmplitudeBlock(name="signal", frequency_hz=frequency_hz),
    observed,
    initial_block_state=initial_at_injection,
    initial_noise_covariance=masked_time_noise,
    num_cycles=2,
)
print("Domain handoff checks passed")

For a WDM campaign, the application can instead create
`transform(observed, "wdm", num_frequency_divisions=8)` and register this same
block with `data_domain="time"`. A block whose own algorithm uses WDM would
request `data_domain="wdm"` and one division count. `L1Data.wdm_grid` is generated
automatically from the resulting coefficients; it is read-only metadata, not a
constructor argument. Both division counts must be positive
and even, and their product must equal the number of underlying time samples.

WDM requires the separately installed local backend and Python 3.13 or newer.
The [domain translation notebook](domain_translation.ipynb) demonstrates that
setup and the two division selectors. Covariance in a new domain need not be
independent per pixel: use its operators or explicitly reject unsupported models.

## 11. A noise-only block uses the same interface

There is no separate NoiseBlock base class. A noise-only block returns
`BlockState(covariance_matrix=..., parameters=...)` with no signal array.

This example samples a shared white-noise variance with a proper inverse-gamma
prior. Given the conditional residual, its conjugate posterior has
`shape = prior_shape + number_of_samples/2` and
`scale = prior_scale + sum_of_squared_residuals/2`. These equations assume fully
observed real time samples and independent Gaussian noise. They do not model gaps
or correlated noise.

The draw is exact and independent of the previous variance, so this block needs
no continuation state. `parameters` records the physical variance; the resulting
`DataCovariance` tells other blocks how to weight their residuals. Initialize this
block with that full covariance and the corresponding physical variance.

In [ ]:
@dataclass(frozen=True)
class WhiteNoiseVarianceBlock:
    name: str = "noise"
    prior_shape: float = 2.0
    prior_scale: float = 0.25

    def __post_init__(self):
        if (
            not np.isfinite([self.prior_shape, self.prior_scale]).all()
            or min(self.prior_shape, self.prior_scale) <= 0
        ):
            raise ValueError("Inverse-gamma prior shape and scale must be positive")

    def _result(self, conditional_residual, variance):
        if conditional_residual.data_domain != "time":
            raise ValueError("This white-noise model requires time-domain inputs")
        return BlockState(
            covariance_matrix=DataCovariance.from_variance(
                conditional_residual, variance
            ),
            parameters={"variance": float(variance)},
        )

    def sample(
        self, conditional_residual, noise_covariance, current_block_state, *, rng
    ):
        if conditional_residual.data_domain != "time":
            raise ValueError("This white-noise model requires time-domain inputs")
        number_of_samples = sum(
            values.size for values in conditional_residual.channel_data.values()
        )
        squared_residual_sum = sum(
            float(values @ values)
            for values in conditional_residual.channel_data.values()
        )
        posterior_shape = self.prior_shape + 0.5 * number_of_samples
        posterior_scale = self.prior_scale + 0.5 * squared_residual_sum
        return self._result(
            conditional_residual, posterior_scale / rng.gamma(posterior_shape)
        )

In [ ]:
initial_noise_result = BlockState(
    covariance_matrix=noise_covariance,
    parameters={"variance": noise_standard_deviation**2},
)
joint_campaign = Wheel(observed, random_seed=9)
joint_campaign.add(
    WhiteNoiseVarianceBlock(),
    initial_block_state=initial_noise_result,
    data_domain="time",
)
joint_campaign.add(
    SineAmplitudeBlock(name="signal", frequency_hz=frequency_hz),
    initial_block_state=initial_at_injection,
    data_domain="time",
)
joint_campaign.run(50)
noise_result = joint_campaign.ledger["noise"]
assert noise_result.template is None
assert noise_result.covariance_matrix is not None
np.testing.assert_array_equal(joint_campaign.contribution("noise")["A"], 0.0)
np.testing.assert_allclose(
    joint_campaign.noise_covariance.noise_variance("A"),
    noise_result.parameters["variance"],
)
print("Last noise variance draw:", noise_result.parameters["variance"])
print("Noise-only block checks passed")

The block that owns the current covariance must republish its complete covariance
on every call. Another block taking ownership replaces that model with a warning;
Wheel does not add noise models together. Publish combined noise components in one
covariance. A joint signal-and-noise block can fill both estimate fields in the
same `BlockState`.

A block that only tracks internal diagnostics may update `self.state` and return
`BlockState()` with no physical contribution.
That contributes no signal and, unless it owns the current covariance, leaves the
noise unchanged. Any previous signal from that block is removed.

## Before you plug in your own model

- Choose the supported channels, observable, TDI convention, domain, and orbit
  needs, then reject unsupported inputs clearly.
- Keep settings, sampler continuation, diagnostics, and live resources on the
  block. Publish complete physical parameters that support deep copying.
- Implement `sample`; supply a complete, consistent `initial_block_state` to
  every `Wheel.add` and `check_block` call. Choose the start in application setup,
  including any optional prior draw, and initialize the model's own `State` too.
- Recompute the conditional likelihood when the residual or noise changes.
- Return a full result after acceptance **and rejection**. Sum all sources modeled
  by your block before publishing its TDI signal.
- Use covariance operators for correlated or translated models. Include the
  determinant when the proposed parameters change the noise normalization.
- Run `check_block` on a separate instance, then independent numerical, posterior,
  and campaign-isolation checks. Collect chains outside the block through a callback.
- Let Wheel handle cross-block residuals and domain conversion.

The [toy fit script](toy_fit.py) provides a compact conjugate sampler, while the
[translation tutorial](domain_translation.ipynb) covers WDM layouts and covariance
semantics in more detail.